# 05 — Multimodal RAG for TALA Claim-Experience Divergence (Day 3B)

## 1. Objective and scope

**Primary objective:** determine whether multimodal evidence (text, image, video, reference)
reveals divergence between TALA's official quality/responsibility claims and externally
observable customer experience -- this notebook demonstrates the retrieval-augmented-generation
(RAG) system built to let a person investigate that question interactively, grounded in real,
cited evidence.

**Explicitly in scope:** claim-experience divergence investigation, open analytical questions,
modality-aware retrieval, Gemini grounding with citation validation, automated (non-human-label)
evaluation.

**Explicitly out of scope:** engagement prediction, manual/human-coder labelling, protected-
characteristic inference from images, any fallback generator or backup vector store.

Full architecture: `docs/multimodal_rag_architecture.md`. Full evaluation narrative:
`docs/multimodal_rag_evaluation.md`.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
pd.set_option("display.max_colwidth", 100)

from src.rag import orchestrator, query_router, rank_fusion, retriever
from src.rag.chroma_store import get_client, get_all_metadata
from src.rag.schemas import COLLECTION_CLAIMS, COLLECTION_TEXT_EVIDENCE, COLLECTION_VISUAL_EVIDENCE, CHROMA_PERSIST_DIR

TABLES = PROJECT_ROOT / "outputs" / "tables"
client = get_client()
print(f"Connected to local persistent ChromaDB at {CHROMA_PERSIST_DIR.relative_to(PROJECT_ROOT)}")

Connected to local persistent ChromaDB at data\vector_db\chroma


## 2. Corpus composition

Built exclusively from existing, already-verified Day 1-3A processed datasets -- nothing here
is freshly scraped. See `docs/multimodal_rag_architecture.md` for the full source table and the
evidence-strength/verification gates applied.

In [2]:
collection_summary = pd.read_csv(TABLES / "rag_collection_summary.csv")
display(collection_summary)

corpus_coverage = pd.read_csv(TABLES / "rag_corpus_coverage.csv")
display(corpus_coverage[corpus_coverage["dimension"] == "source_type"])

,collection,n_final,n_upserted_this_run,n_removed_stale_this_run,n_missing_asset_rejected,n_embed_failed_rejected
0,tala_claims,37,37,0,0,0
1,tala_text_evidence,662,662,0,0,0
2,tala_visual_evidence,76,76,0,0,0


,collection,dimension,value,n_rows
1,tala_claims,source_type,official_claim,37
7,tala_text_evidence,source_type,creator_strategy,61
8,tala_text_evidence,source_type,customer_experience,95
9,tala_text_evidence,source_type,official_claims,37
10,tala_text_evidence,source_type,official_reference,461
11,tala_text_evidence,source_type,official_video,8
18,tala_visual_evidence,source_type,official_product_image,24
19,tala_visual_evidence,source_type,official_video,52


**Documented gap, not silently dropped:** `data/processed/reference_images.csv` (584 rows)
has no downloaded local asset for any row, so none are CLIP-embedded into `tala_visual_evidence`
-- they remain a citation-only catalogue, not indexed visual evidence.

## 3. Chroma collections

Three persistent collections at `data/vector_db/chroma/`, cosine distance, deterministic IDs
(`claim::<id>`, `text::<source_type>::<raw_id>`, `visual::<modality>::<raw_id>`) so the index
builder is idempotent on rerun.

In [3]:
for name in (COLLECTION_CLAIMS, COLLECTION_TEXT_EVIDENCE, COLLECTION_VISUAL_EVIDENCE):
    collection = client.get_collection(name)
    print(f"{name}: {collection.count()} rows")

modality_coverage = pd.read_csv(TABLES / "rag_modality_coverage.csv")
display(modality_coverage)

tala_claims: 37 rows


tala_text_evidence: 662 rows
tala_visual_evidence: 76 rows


,collection,modality,n_rows
0,tala_text_evidence,text,654
1,tala_text_evidence,video_summary,8
2,tala_visual_evidence,image,24
3,tala_visual_evidence,video_frame,52


## 4. Modality routing

Deterministic, inspectable, reuses the authoritative Day 3A visual-groundability rule
(`configs/fusion_rules.yaml`) -- no LLM secretly decides what gets searched.

In [4]:
for category in ("materials", "packaging", "labour", "emissions", "manufacturing"):
    modalities = query_router.route_modalities_for_claim(category)
    print(f"{category:15s} -> {modalities}")

materials       -> ['text', 'reference', 'image', 'video']
packaging       -> ['text', 'reference', 'image', 'video']
labour          -> ['text', 'reference']
emissions       -> ['text', 'reference']
manufacturing   -> ['text', 'reference']


In [5]:
for question in [
    "What evidence challenges TALA's durability claims?",
    "Which responsibility claims lack independent validation?",
    "How does TALA's creator partnership intent differ from competitors?",
]:
    intents = query_router.classify_intents(question)
    modalities = query_router.route_modalities_for_intents(intents)
    print(f"Q: {question}\n  intents={intents}\n  modalities={modalities}\n")

Q: What evidence challenges TALA's durability claims?
  intents=['durability_care', 'claim_validation']
  modalities=['text', 'image', 'video', 'reference']

Q: Which responsibility claims lack independent validation?
  intents=['sustainability_responsibility', 'evidence_gap_analysis', 'claim_validation']
  modalities=['text', 'image', 'video', 'reference']

Q: How does TALA's creator partnership intent differ from competitors?
  intents=['creator_strategy', 'competitor_comparison']
  modalities=['text']



**Modality-routing accuracy** (`outputs/tables/rag_query_routing_evaluation.csv`):
every one of the 37 real claims is checked against the authoritative groundability config.

In [6]:
routing_eval = pd.read_csv(TABLES / "rag_query_routing_evaluation.csv")
print(f"{routing_eval['routing_correct'].sum()}/{len(routing_eval)} claims routed correctly")
display(routing_eval.head(10))

37/37 claims routed correctly


,claim_id,claim_category,expected_visually_groundable,actual_routed_visual,routing_correct
0,c_OC_0002_00,labour,False,False,True
1,c_OC_0002_01,labour,False,False,True
2,c_OC_0003_01,materials,True,True,True
3,c_OC_0004_00,materials,True,True,True
4,c_OC_0005_00,labour,False,False,True
5,c_OC_0005_01,labour,False,False,True
6,c_OC_0007_00,labour,False,False,True
7,c_OC_0007_01,labour,False,False,True
8,c_OC_0008_00,labour,False,False,True
9,c_OC_0008_01,labour,False,False,True


## 5. Retrieval pipeline

Text retrieval (sentence-transformers `all-MiniLM-L6-v2`) and visual retrieval (CLIP
`openai/clip-vit-base-patch32` text-tower query against CLIP image embeddings), plus direct
Day 3A claim-evidence links looked up by metadata (not similarity).

In [7]:
text_results = retriever.retrieve_text_evidence(client, "durability of the fabric after washing", n_results=5)
for r in text_results:
    print(r["id"], round(r["similarity"], 3), r["metadata"]["source_type"])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

text::reference_chunk::ref_tala_701_chunk_001 0.499 official_reference
text::reference_chunk::ref_tala_700_chunk_001 0.494 official_reference
text::reference_chunk::ref_girlfriend_collective_006_chunk_002 0.441 official_reference
text::reference_chunk::ref_girlfriend_collective_007_chunk_002 0.441 official_reference
text::reference_chunk::ref_tala_015_chunk_003 0.409 official_reference


In [8]:
visual_results = retriever.retrieve_visual_evidence(client, "a person wearing activewear leggings", n_results=5)
for r in visual_results:
    print(r["id"], round(r["similarity"], 3), r["metadata"]["modality"], r["metadata"].get("video_role", ""))

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

visual::video_frame::tala_video_105_quarter_111 0.316 video_frame other
visual::image::tala_004 0.312 image 
visual::image::oner_active_006 0.301 image 
visual::image::oner_active_003 0.301 image 
visual::image::tala_003 0.3 image 


### Video evidence is genuinely temporal, not one thumbnail

Every video result carries video identity, a representative frame position (opening/interval/
quarter/half/three_quarter/closing), an exact timestamp, video role, and is drawn only from the
8 genuinely processed TALA videos (the other 61 discovered videos are metadata-only leads and
are never treated as video evidence).

In [9]:
video_frames = [r for r in visual_results if r["metadata"]["modality"] == "video_frame"]
if video_frames:
    display(pd.DataFrame([r["metadata"] for r in video_frames])[["video_id", "opening_middle_closing_position", "timestamp_seconds", "video_role"]])
else:
    print("No video_frame results in this particular sample query -- see the text+image+video demo case below for a guaranteed example.")

,video_id,opening_middle_closing_position,timestamp_seconds,video_role
0,tala_video_105,middle,3.7,other


## 6. Rank fusion

Reciprocal-rank fusion (k=60) across independently-retrieved lists, plus deterministic bonuses
for direct claim-evidence links, evidence strength, and source independence. NLI is never used
as the relevance/stance authority -- Day 3A already established it is auxiliary-only.

In [10]:
result = orchestrator.investigate_claim("c_OC_0003_01", n_results=8, generate=False)
trace_rows = [{
    "id": c["id"], "collection": c["collection"], "final_rank": c.get("final_rank"),
    "final_score": c.get("final_score"), "rrf_contribution": c["trace"]["rank_fusion_contribution"],
    "direct_link_bonus": c["trace"].get("direct_link_bonus"), "exclusion_reason": c["trace"]["exclusion_reason"],
} for c in result["fused_candidates"]]
display(pd.DataFrame(trace_rows))

,id,collection,final_rank,final_score,rrf_contribution,direct_link_bonus,exclusion_reason
0,text::customer_experience::ce_press_0016,tala_text_evidence,0.0,1.217522,0.032522,1.0,NaN
1,text::customer_experience::ce_rev_0002,tala_text_evidence,1.0,1.201393,0.016393,1.0,NaN
2,text::reference_chunk::ref_tala_014_chunk_017,tala_text_evidence,2.0,1.182266,0.032266,1.0,NaN
3,visual::image::tala_005,tala_visual_evidence,3.0,1.172522,0.032522,1.0,NaN
4,visual::video_frame::tala_video_104_half_152,tala_visual_evidence,4.0,1.171319,0.031319,1.0,NaN
5,text::video_summary::tala_video_104,tala_text_evidence,5.0,1.166393,0.016393,1.0,NaN
6,text::creator_strategy::cs_0021,tala_text_evidence,6.0,1.156393,0.016393,1.0,NaN
7,visual::video_frame::tala_video_104_opening_0,tala_visual_evidence,7.0,1.156393,0.016393,1.0,NaN
8,visual::video_frame::tala_video_104_quarter_76,tala_visual_evidence,NaN,1.156393,0.016393,1.0,max_results_exceeded
9,visual::video_frame::tala_video_104_three_quarter_229,tala_visual_evidence,NaN,1.156393,0.016393,1.0,max_results_exceeded


## 7. Gemini grounding

`google-genai` is the only generation path. The prompt contains only the question, the selected
claim and its existing Day 3A fusion label (never re-decided by Gemini), the retrieved evidence
with explicit evidence IDs, and grounding instructions. If `GEMINI_API_KEY`/`GEMINI_MODEL` are
missing or the API call fails, generation stops -- there is no fallback.

In [11]:
from src.rag.gemini_generator import GeminiConfigError, GeminiGenerationError
try:
    demo = orchestrator.investigate_claim("c_OC_0003_01", n_results=8, generate=True)
    print("MODEL USED:", demo["generation"]["_model_used"])
    print("\nCONCISE ANSWER:\n", demo["generation"]["concise_answer"])
    print("\nCITATION VALIDATION PASSED:", demo["validation"]["passed"])
except (GeminiConfigError, GeminiGenerationError) as exc:
    print(f"Gemini call did not succeed in this run (shown honestly, no fallback substituted): {exc}")

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


MODEL USED: gemini-3.1-flash-lite

CONCISE ANSWER:
 The claim aligns with TALA's internal statements regarding their production and sourcing practices, which are supported by marketing materials and brand reviews.

CITATION VALIDATION PASSED: True


## 8. Citation validation

Every Gemini response is checked automatically: cited IDs must exist in the retrieved context,
the production fusion label must be preserved, self-reported evidence must never be called
independent, and the response must never claim to have inspected evidence beyond what was
supplied (e.g. a full video when only sampled frames were given).

In [12]:
citation_eval = pd.read_csv(TABLES / "rag_citation_validation.csv")
display(citation_eval)

,demo_case,claim_id,passed,failures,cited_id_count,check_citations_exist_in_context,check_referenced_assets_resolve,check_production_label_preserved,check_self_reported_not_called_independent,check_no_overclaimed_inspection
0,text_image_video_claim,c_OC_0003_01,True,NaN,7,True,True,True,True,True
1,mixed_claim_explanation,c_OC_0010_00,True,NaN,6,True,True,True,True,True
2,insufficient_evidence_explanation,c_OC_0002_00,True,NaN,7,True,True,True,True,True


## 9. Automated evaluation (no human-label evaluation)

Every metric below is derived from existing structural relationships (claim-evidence links,
claim categories, modality eligibility, stance, production fusion labels) -- no new manual-
label CSV was created, no coder_1/coder_2/adjudication workflow exists, and Gemini is never
used to grade its own answers.

In [13]:
retrieval_eval = pd.read_csv(TABLES / "rag_retrieval_evaluation.csv")
display(retrieval_eval)

,collection,n_rows,n_unique_ids,duplicate_id_rate,metric_group,field,n_present,n_total,completeness_pct,metric,value,n_missing,n_total_attempted
0,tala_claims,37.0,37.0,0.0,collection_integrity,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,tala_text_evidence,662.0,662.0,0.0,collection_integrity,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,tala_visual_evidence,76.0,76.0,0.0,collection_integrity,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,tala_claims,NaN,NaN,NaN,metadata_completeness,claim_id,37.0,37.0,100.0,NaN,NaN,NaN,NaN
4,tala_claims,NaN,NaN,NaN,metadata_completeness,claim_category,37.0,37.0,100.0,NaN,NaN,NaN,NaN
5,tala_claims,NaN,NaN,NaN,metadata_completeness,fusion_label,37.0,37.0,100.0,NaN,NaN,NaN,NaN
6,tala_claims,NaN,NaN,NaN,metadata_completeness,confidence,37.0,37.0,100.0,NaN,NaN,NaN,NaN
7,tala_claims,NaN,NaN,NaN,metadata_completeness,presentation_restriction,37.0,37.0,100.0,NaN,NaN,NaN,NaN
8,tala_text_evidence,NaN,NaN,NaN,metadata_completeness,evidence_id,662.0,662.0,100.0,NaN,NaN,NaN,NaN
9,tala_text_evidence,NaN,NaN,NaN,metadata_completeness,modality,662.0,662.0,100.0,NaN,NaN,NaN,NaN


In [14]:
latency = pd.read_csv(TABLES / "rag_latency_summary.csv")
display(latency)

,stage,latency_seconds
0,text_retrieval_cold_incl_model_load,0.0315
1,text_retrieval_warm,0.0345
2,visual_retrieval_cold_incl_model_load,6.7073
3,visual_retrieval_warm,0.0700
4,end_to_end_retrieval_no_generation_warm,0.1195


## 10. Demonstration cases

At least one text+image+video claim, one mixed-label claim explanation, and one insufficient-
evidence explanation.

In [15]:
demo_cases = pd.read_csv(TABLES / "rag_demo_cases.csv")
display(demo_cases)

,demo_case,claim_id,true_fusion_label,n_supports,n_challenges,n_context,modalities_in_final_evidence,citation_validation_passed,outcome,api_outcome,model,generated_at
0,text_image_video_claim,c_OC_0003_01,aligned,4,0,4,image;text;video_frame;video_summary,True,success,ok,"gemini-3.1-flash-lite, prompt_version=day3b_v1",2026-09-19T20:04:04.415938+00:00
1,mixed_claim_explanation,c_OC_0010_00,mixed,2,1,5,text,True,success,ok,"gemini-3.1-flash-lite, prompt_version=day3b_v1",2026-09-19T20:04:04.415938+00:00
2,insufficient_evidence_explanation,c_OC_0002_00,insufficient_evidence,0,0,8,text,True,success,ok,"gemini-3.1-flash-lite, prompt_version=day3b_v1",2026-09-19T20:04:04.415938+00:00


## 11. Limitations

- `reference_images.csv`'s 584 rows have no downloaded local asset and are excluded from visual
  evidence -- a real, documented corpus gap, not a Day 3B omission.
- Video evidence is genuinely temporal (sampled frames + motion/scene features) but is **not**
  full raw-video understanding -- the prompt and validator both enforce this distinction.
- Official reference/claim evidence **may be self-reported**; the system labels this explicitly.
- Absence of independent evidence does not prove a claim false -- `insufficient_evidence` claims
  are surfaced as evidence gaps, never as negative conclusions.
- Automated evaluation measures structural correctness (recall, routing accuracy, citation
  validity) -- it does not substitute for human semantic judgement of explanation quality, and
  human-label evaluation was deliberately excluded from this project's scope throughout.
- Live Gemini demo-case generation is subject to the API's free-tier rate/quota limits; a 429
  or 503 response is shown honestly (see `rag_demo_cases.csv`'s `api_outcome` column) rather
  than masked with a substitute answer.

## 12. Final GO/NO-GO

In [16]:
go_no_go = pd.read_csv(TABLES / "rag_go_no_go.csv")
display(go_no_go)

print("=" * 70)
print("NOTEBOOK 05 COMPLETE -- Day 3B Multimodal RAG")
print("=" * 70)
for _, row in go_no_go.iterrows():
    print(f"  {row['gate']:<45} {row['status']}")

,gate,status,evidence
0,no_duplicate_ids,GO,"duplicate_id_rate per collection: [0.0, 0.0, 0.0]"
1,no_missing_visual_paths,GO,0 missing_asset_rejected in visual collection build
2,text_retrieval_works,GO,recall@10 over 21 claims: 0.7619047619047619
3,modality_routing_correct,GO,37/37 claims routed correctly
4,text_image_video_demo,GO,outcome: ['success'].
5,mixed_claim_demo,GO,outcome: ['success'].
6,insufficient_evidence_demo,GO,outcome: ['success'].
7,citations_resolve_to_retrieved_evidence,GO,3/3 demo generations passed citation validation
8,production_fusion_labels_preserved,GO,all demo-case generations preserved the true fusion label
9,no_fallback_or_backup_implementation,GO,"single ChromaDB store, single Gemini generator; no alternative vector store or local-LLM fallbac..."


NOTEBOOK 05 COMPLETE -- Day 3B Multimodal RAG
  no_duplicate_ids                              GO
  no_missing_visual_paths                       GO
  text_retrieval_works                          GO
  modality_routing_correct                      GO
  text_image_video_demo                         GO
  mixed_claim_demo                              GO
  insufficient_evidence_demo                    GO
  citations_resolve_to_retrieved_evidence       GO
  production_fusion_labels_preserved            GO
  no_fallback_or_backup_implementation          GO
  no_engagement_prediction                      GO
  no_human_labelling_dependency                 GO
